# 🎬 IMDB 电影数据探索性分析报告

---
<div style="text-align:center;">

**课程名称**：数据分析与可视化

**学生姓名**：XXX

**学    号**：XXX

**提交日期**：2026 年 6 月

</div>

---

## 📋 目录

1. [数据读取与分析目标](#1-数据读取与分析目标)
2. [数据基本情况](#2-数据基本情况)
3. [数据质量检查](#3-数据质量检查)
4. [数据清洗与字段转换](#4-数据清洗与字段转换)
5. [描述性统计分析](#5-描述性统计分析)
6. [可视化分析](#6-可视化分析)
   - 6.1 电影类型分布
   - 6.2 年度电影产出趋势
   - 6.3 电影时长分布
   - 6.4 IMDB 评分分布
   - 6.5 评分与票房关系
   - 6.6 各年份评分对比
   - 6.7 高产导演 TOP10
   - 6.8 数值特征相关性
   - 6.9 电影类型：口碑 vs 票房
7. [主要结论](#7-主要结论)


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

# ========== 清除 matplotlib 字体缓存（解决中文方块问题） ==========
_cache_cleared = False
for _cache_dir in [os.path.expanduser('~/.matplotlib'),
                   os.path.expanduser('~/.cache/matplotlib')]:
    try:
        for _fn in os.listdir(_cache_dir):
            if 'font' in _fn.lower():
                os.remove(os.path.join(_cache_dir, _fn))
                _cache_cleared = True
    except:
        pass

if _cache_cleared:
    fm._load_fontmanager(try_read_cache=False)
    print('Font cache cleared and rebuilt.')

# ==================== 字体配置（防止中文乱码） ====================
# 优先使用 Noto Sans SC（Google 专业中文字体，字形完整无方块）
# 备选：Microsoft YaHei, SimHei, WenQuanYi Micro Hei
_chinese_fonts = ['Microsoft YaHei', 'SimHei', 'Noto Sans SC', 'STXihei',
                  'FangSong', 'KaiTi', 'WenQuanYi Micro Hei', 'WenQuanYi Zen Hei']

_selected_font = None
_available = {f.name for f in fm.fontManager.ttflist}
for _f in _chinese_fonts:
    if _f in _available:
        _selected_font = _f
        break

if _selected_font is None:
    # 暴力搜索任何支持 CJK 的字体
    for _f in fm.fontManager.ttflist:
        try:
            if any('一' <= c <= '鿿' for c in _f.name):
                _selected_font = _f.name
                break
        except:
            pass

if _selected_font is None:
    _selected_font = 'sans-serif'
    print("WARNING: No Chinese font found! Using sans-serif fallback.")

print(f"Using font: {_selected_font}")

plt.rcParams['font.sans-serif'] = [_selected_font, 'DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.serif'] = [_selected_font, 'DejaVu Serif', 'Times New Roman']
plt.rcParams['font.monospace'] = [_selected_font, 'DejaVu Sans Mono', 'Courier New']
plt.rcParams['axes.unicode_minus'] = False

# 强制重建字体缓存，确保中文字体被正确加载
try:
    fm._load_fontmanager(try_read_cache=False)
except:
    pass

# ==================== 全局样式（高级配色 + 排版） ====================
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.15)

# 自定义高级调色板
C = {
    'navy':    '#1B3A5C',
    'coral':   '#E8583D',
    'teal':    '#2E8B7C',
    'gold':    '#D4993D',
    'slate':   '#4A5568',
    'blue':    '#3B6FB6',
    'red':     '#C0392B',
    'green':   '#27AE60',
    'purple':  '#6C3A96',
    'orange':  '#E67E22',
    'pink':    '#C2185B',
}

PAL_BLUES   = sns.color_palette("Blues_r", 15)
PAL_REDS    = sns.color_palette("Reds_r", 15)
PAL_VIRIDIS = sns.color_palette("viridis", 15)
PAL_MAGMA   = sns.color_palette("magma", 10)
PAL_RDBU    = sns.diverging_palette(240, 10, s=85, l=45, n=15, center='light')
PAL_COOL    = sns.color_palette("coolwarm", 15)

plt.rcParams.update({
    'figure.dpi':         180,
    'figure.figsize':     (10, 6),
    'figure.facecolor':   'white',
    'axes.titlesize':     15,
    'axes.titleweight':   'bold',
    'axes.labelsize':     12,
    'axes.labelweight':   '600',
    'xtick.labelsize':    9.5,
    'ytick.labelsize':    9.5,
    'legend.fontsize':    10,
    'legend.title_fontsize': 11,
    'lines.linewidth':    2.2,
    'patch.edgecolor':    'white',
    'patch.linewidth':    0.4,
})

# 确保 display 可用
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

print("All libraries loaded successfully.")


## 1. 数据读取与分析目标

### 数据来源
IMDB（Internet Movie Database）公开电影数据集，收录约 1000 部电影详细信息。

### 分析目标
通过数据清洗、描述性统计与可视化，探究电影类型格局、时间趋势、评分分布、票房驱动因素以及创作者生产力等。

### 数据字段

| 字段（英文） | 字段（中文） | 类型 | 说明 |
|---|---|---|---|
| Rank | 排名 | int | 电影综合排名 |
| Title | 片名 | str | 电影名称 |
| Genre | 类型 | str | 电影类型（多个以逗号分隔） |
| Description | 简介 | str | 剧情概要 |
| Director | 导演 | str | 导演姓名 |
| Actors | 演员 | str | 主演列表 |
| Year | 上映年份 | int | 上映年份 |
| Runtime (Minutes) | 时长（分钟） | int | 电影时长 |
| Rating | 用户评分 | float | IMDB 用户评分（1-10） |
| Votes | 投票数 | int | IMDB 参与评分的用户数 |
| Revenue (Millions) | 票房（百万美元） | float | 全球票房收入 |
| Metascore | 媒体评分 | float | 专业媒体综合评分（0-100） |


In [ ]:

df = pd.read_csv('IMDB-Movie-Data.csv')

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print()

print("=" * 60)
print("First 5 rows preview:")
print("=" * 60)
display(df.head())

print()
print("=" * 60)
print("Data types:")
print("=" * 60)
display(df.dtypes.to_frame('Data Type'))


## 2. 数据质量检查

在进行任何分析之前，先对数据质量进行全面排查：缺失值、重复值、异常值、格式问题等。


In [ ]:

print("=" * 60)
print("Missing Values")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
})
display(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))

print()
print("=" * 60)
print("Duplicate Check")
print("=" * 60)
print(f"Duplicate rows: {df.duplicated().sum()}")

print()
print("=" * 60)
print("Numerical Summary")
print("=" * 60)
display(df.describe())

print()
print("=" * 60)
print("Extreme Values Check")
print("=" * 60)
print("Bottom 5 movies (by Rating):")
display(df.nsmallest(5, 'Rating')[['Title', 'Year', 'Rating', 'Votes']])

abnormal = df[(df['Runtime (Minutes)'] < 70) | (df['Runtime (Minutes)'] > 180)]
print(f"Abnormal runtime (<70 or >180 min): {len(abnormal)} movies ({len(abnormal)/len(df)*100:.1f}%)")
display(abnormal[['Title', 'Year', 'Runtime (Minutes)', 'Rating']])


## 3. 数据清洗与字段转换

| 问题 | 处理方法 | 理由 |
|---|---|---|
| Revenue (Millions) 缺失 128 个（12.8%） | 按年份分组中位数填充 | 票房偏态分布，中位数比均值稳健 |
| Metascore 缺失 64 个（6.4%） | 全局均值填充 | 接近正态，均值合理 |
| Genre 为多值字段 | 按逗号拆分，展开用于类型频次统计 | 一部电影属于多个类型需分别计数 |


In [ ]:

df_clean = df.copy()

# 1. 按年份分组，用中位数填充票房缺失值
df_clean['Revenue (Millions)'] = (
    df_clean.groupby('Year')['Revenue (Millions)']
    .transform(lambda x: x.fillna(x.median()))
)
# 若整年缺失，再用全局中位数
df_clean['Revenue (Millions)'] = df_clean['Revenue (Millions)'].fillna(
    df_clean['Revenue (Millions)'].median()
)

# 2. Metascore 用均值填充
df_clean['Metascore'] = df_clean['Metascore'].fillna(df_clean['Metascore'].mean())

# 3. 拆分电影类型
df_clean['Genre_List'] = df_clean['Genre'].str.split(',')
all_genres = []
for gl in df_clean['Genre_List']:
    all_genres.extend([g.strip() for g in gl])
genre_count = pd.Series(all_genres).value_counts()

print("Data cleaning complete!")
print(f"  Revenue missing after: {df_clean['Revenue (Millions)'].isnull().sum()}")
print(f"  Metascore missing after: {df_clean['Metascore'].isnull().sum()}")
print(f"  Unique genres: {len(genre_count)}")
print(f"  Top 5 genres: {dict(genre_count.head(5))}")


## 4. 描述性统计分析


In [ ]:

numeric_cols = ['Year', 'Runtime (Minutes)', 'Rating', 'Votes',
                'Revenue (Millions)', 'Metascore']

stats = df_clean[numeric_cols].describe().round(2)
stats.loc['skew'] = df_clean[numeric_cols].skew().round(2)
stats.loc['kurtosis'] = df_clean[numeric_cols].kurtosis().round(2)

stats = stats.rename(index={
    'count': 'Sample Size',
    'mean': 'Mean',
    'std': 'Std Dev',
    'min': 'Min',
    '25%': '25th Pctl',
    '50%': 'Median',
    '75%': '75th Pctl',
    'max': 'Max',
    'skew': 'Skewness',
    'kurtosis': 'Kurtosis'
})

display(stats)

print()
print("Key observations:")
print(f"  Year range: {int(df_clean['Year'].min())} - {int(df_clean['Year'].max())}")
print(f"  Avg rating: {df_clean['Rating'].mean():.2f} (std {df_clean['Rating'].std():.2f})")
print(f"  Avg runtime: {df_clean['Runtime (Minutes)'].mean():.0f} min")
print(f"  Avg revenue: ${df_clean['Revenue (Millions)'].mean():.1f}M")
print(f"  Median revenue: ${df_clean['Revenue (Millions)'].median():.1f}M")
print(f"  Revenue skewness: {df_clean['Revenue (Millions)'].skew():.2f} (heavily right-skewed)")


## 5. 可视化分析

---

### 5.1 电影类型分布 TOP15

**分析问题**：哪些电影类型最受市场青睐？


In [ ]:

# ============ Chart 1: 电影类型分布（Lollipop Chart） ============
fig, ax = plt.subplots(figsize=(13, 8))

genres_top = genre_count.head(15)
n = len(genres_top)
y_pos = range(n)

# 配色：渐变色
colors = sns.color_palette("Blues_r", n_colors=n)

# 棒棒糖线条（茎）
for i, (val, color) in enumerate(zip(genres_top.values, colors)):
    ax.plot([0, val], [i, i], '-', color=color, linewidth=2.5, alpha=0.7, zorder=2)

# 棒棒糖头部（散点）
ax.scatter(genres_top.values, y_pos, s=180, c=colors, edgecolor='white',
           linewidth=1.2, zorder=5)

# 数据标签
for i, val in enumerate(genres_top.values):
    ax.text(val + 10, i, f'{val}', va='center', fontsize=11, fontweight='bold',
            color=C['navy'])

# 类型标签
ax.set_yticks(y_pos)
ax.set_yticklabels(genres_top.index, fontsize=12)
ax.invert_yaxis()

# 横纵坐标轴标签
ax.set_xlabel('电影数量（部）', fontsize=13, fontweight='bold')
ax.set_ylabel('电影类型', fontsize=13, fontweight='bold')
ax.set_title('电影类型分布 TOP 15', fontsize=16, fontweight='bold', pad=15)

# 标注第一
ax.annotate('Drama dominates\nwith {0} movies'.format(genres_top.values[0]),
            xy=(genres_top.values[0], 0),
            xytext=(genres_top.values[0] * 0.55, 3),
            fontsize=10, color=C['coral'], fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C['coral'], lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF0ED', edgecolor=C['coral'], alpha=0.85))

ax.set_xlim(0, genres_top.values.max() * 1.25)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('1_电影类型分布.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print("Drama, Action, Comedy together account for over 50% of the dataset.")


### 5.2 年度电影产出趋势（2006 - 2016）

**分析问题**：过去十年电影产量如何变化？


In [ ]:

# ============ Chart 2: 年度电影数量趋势 ============
fig, ax = plt.subplots(figsize=(13, 7))

year_count = df_clean['Year'].value_counts().sort_index()
years = year_count.index.astype(float)
counts = year_count.values.astype(float)

# 1. 面积渐变填充
ax.fill_between(years, counts, alpha=0.18, color=C['blue'], zorder=1)

# 2. 主折线
ax.plot(years, counts, '-', color=C['blue'], linewidth=3.5, marker='o',
        markersize=11, markerfacecolor='white', markeredgewidth=2.8,
        markeredgecolor=C['blue'], zorder=4, label='年度电影数量')

# 3. 趋势线（二次多项式拟合）
z = np.polyfit(years, counts, 2)
p = np.poly1d(z)
x_smooth = np.linspace(years.min(), years.max(), 200)
ax.plot(x_smooth, p(x_smooth), '--', color=C['coral'], linewidth=2.2, alpha=0.8,
        zorder=3, label='二次趋势线')

# 4. 数据标签
for x, y in zip(years, counts):
    ax.text(x, y + 12, f'{int(y)}', ha='center', fontsize=10, fontweight='bold',
            color=C['slate'])

# 5. 标注增长
ax.annotate('+{:.0f}% growth\\n2006 to 2016'.format((counts[-1]/counts[0]-1)*100),
            xy=(years[-1], counts[-1]),
            xytext=(years[5], counts.max() + 55),
            fontsize=12, ha='center', fontweight='bold', color=C['coral'],
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=C['coral'], alpha=0.85),
            arrowprops=dict(arrowstyle='->', color=C['coral'], lw=1.8))

# 横纵坐标轴标签
ax.set_xlabel('上映年份', fontsize=13, fontweight='bold')
ax.set_ylabel('电影数量（部）', fontsize=13, fontweight='bold')
ax.set_title('2006-2016 年电影产出趋势', fontsize=16, fontweight='bold', pad=15)
ax.set_xticks(years.astype(int))
ax.set_ylim(0, counts.max() * 1.45)
ax.legend(fontsize=11, loc='upper left', frameon=True, facecolor='white', edgecolor='#ddd')

sns.despine()
plt.tight_layout()
plt.savefig('2_年度电影趋势.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print("Clear upward trend: from 44 movies (2006) to 297 (2016), a {:.0f}% increase.".format(
    (counts[-1]/counts[0]-1)*100))


### 5.3 电影时长分布

**分析问题**：院线电影时长呈现怎样的分布规律？


In [ ]:

# ============ Chart 3: 电影时长分布（Histogram + KDE + Rug） ============
fig, ax = plt.subplots(figsize=(13, 7))

runtime = df_clean['Runtime (Minutes)']
mean_r = runtime.mean()
median_r = runtime.median()
q25, q75 = runtime.quantile(0.25), runtime.quantile(0.75)

# 直方图 + KDE
sns.histplot(runtime, bins=30, kde=True, color=C['blue'],
             alpha=0.45, edgecolor='white', linewidth=0.6, ax=ax,
             label='直方图 + KDE 密度曲线')

# 地毯图（rug plot）—— 底部小刻度展示每个数据点
sns.rugplot(runtime, color=C['slate'], alpha=0.25, height=0.04, ax=ax)

# 均值线
ax.axvline(mean_r, color=C['coral'], linestyle='--', linewidth=2.8,
           label=f'平均值：{mean_r:.0f} 分钟')
# 中位数线
ax.axvline(median_r, color=C['teal'], linestyle='-.', linewidth=2.8,
           label=f'中位数：{median_r:.0f} 分钟')

# IQR 区间标注
ax.axvspan(90, 150, alpha=0.07, color=C['gold'])
ax.text(120, ax.get_ylim()[1] * 0.93, 'Typical Range\n90 - 150 min',
        ha='center', fontsize=10, fontweight='bold', color='#B8860B')

# Q1/Q3 标注
for q, label, ls in [(q25, 'Q1', ':'), (q75, 'Q3', ':')]:
    ax.axvline(q, color='gray', linestyle=ls, linewidth=1.2, alpha=0.5)
    ax.text(q + 1, ax.get_ylim()[1] * 0.5, f'{label}\n{q:.0f}min', fontsize=8, color='gray')

# 横纵坐标轴标签
ax.set_xlabel('电影时长（分钟）', fontsize=13, fontweight='bold')
ax.set_ylabel('电影数量（部）', fontsize=13, fontweight='bold')
ax.set_title('电影时长分布', fontsize=16, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='upper right', frameon=True, facecolor='white', edgecolor='#ddd')

sns.despine()
plt.tight_layout()
plt.savefig('3_电影时长分布.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f"Runtime approx. normal distribution, centered at {mean_r:.0f} min.")
print(f"90% of movies fall between {runtime.quantile(0.05):.0f} - {runtime.quantile(0.95):.0f} min.")


### 5.4 IMDB 用户评分分布

**分析问题**：IMDB 用户评分服从什么分布？是否有评分偏差？


In [ ]:

# ============ Chart 4: IMDB评分分布（双面板） ============
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

rating = df_clean['Rating']
mean_rt = rating.mean()
median_rt = rating.median()

# ---- Panel A: Histogram + KDE ----
ax1 = axes[0]

sns.histplot(rating, bins=26, kde=True, color=C['teal'],
             alpha=0.5, edgecolor='white', linewidth=0.6, ax=ax1)

# 均值 + 中位数
ax1.axvline(mean_rt, color=C['coral'], linestyle='--', linewidth=2.8,
            label=f'平均值 = {mean_rt:.2f}')
ax1.axvline(median_rt, color=C['navy'], linestyle='-.', linewidth=2.8,
            label=f'中位数 = {median_rt:.2f}')

# 标注评分区间
segments = [(1, 5, 'Low\n(1-5)'), (5, 7, 'Medium\n(5-7)'), (7, 9.1, 'High\n(7-9)')]
seg_colors = [C['coral'], C['gold'], C['teal']]
for (lo, hi, label), sc in zip(segments, seg_colors):
    cnt = ((rating >= lo) & (rating < hi)).sum()
    ax1.axvspan(lo, hi, alpha=0.08, color=sc)
    mid = (lo + hi) / 2
    ax1.text(mid, ax1.get_ylim()[1] * 0.22, f'{label}\n{cnt} ({cnt/len(rating)*100:.1f}%)',
             ha='center', fontsize=9, fontweight='bold', color=sc)

ax1.set_xlabel('IMDB 评分', fontsize=12, fontweight='bold')
ax1.set_ylabel('电影数量（部）', fontsize=12, fontweight='bold')
ax1.set_title('A. 评分分布直方图', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10, loc='upper left')

# ---- Panel B: Violin + Box + Swarm overlay ----
ax2 = axes[1]

# 小提琴图
parts = ax2.violinplot(rating, positions=[0], vert=True, widths=0.7,
                        showmeans=True, showmedians=True)
for pc in parts['bodies']:
    pc.set_facecolor(C['blue'])
    pc.set_alpha(0.45)

# 箱线图叠加上去
bp = ax2.boxplot(rating, positions=[0], widths=0.18, patch_artist=True,
                  medianprops={'color': C['navy'], 'linewidth': 2.5},
                  whiskerprops={'linewidth': 1.5},
                  capprops={'linewidth': 1.5},
                  flierprops={'marker': 'o', 'markerfacecolor': C['coral'],
                             'markersize': 5, 'alpha': 0.4})
bp['boxes'][0].set_facecolor('white')
bp['boxes'][0].set_alpha(0.7)

# 被选中的散点覆盖
np.random.seed(42)
jitter = np.random.uniform(-0.12, 0.12, len(rating))
ax2.scatter(jitter, rating, alpha=0.35, s=12, color=C['slate'], zorder=5)

# 统计量标注
stats_text = (
    f"Mean = {mean_rt:.2f}\n"
    f"Median = {median_rt:.2f}\n"
    f"Std = {rating.std():.2f}\n"
    f"Skew = {rating.skew():.2f}\n"
    f"Q1 = {rating.quantile(0.25):.2f}\n"
    f"Q3 = {rating.quantile(0.75):.2f}"
)
ax2.text(1.4, rating.median(), stats_text, fontsize=10, fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#ccc', alpha=0.9),
         verticalalignment='center')

ax2.set_ylabel('IMDB 评分', fontsize=12, fontweight='bold')
ax2.set_xlabel('全部电影', fontsize=12, fontweight='bold')
ax2.set_title('B. 小提琴图 + 箱线图（含散点）', fontsize=14, fontweight='bold')
ax2.set_xticklabels([''])
ax2.set_ylim(0.5, 10)

plt.tight_layout()
plt.savefig('4_IMDB评分分布.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f"Ratings concentrate in 6-8 range, slightly left-skewed.")
print(f"Median ({median_rt:.2f}) > Mean ({mean_rt:.2f}) indicates a few low-rated movies pull the average down.")


### 5.5 评分与票房关系

**分析问题**：IMDB 评分能否预测票房？口碑与商业成功的关联有多强？


In [ ]:

# ============ Chart 5: 评分 vs 票房（Hexbin + 边缘直方图） ============
df_plot = df_clean[df_clean['Revenue (Millions)'] > 0]
corr_rr = df_plot['Rating'].corr(df_plot['Revenue (Millions)'])

# 使用 JointGrid 创建高级联合分布图
g = sns.JointGrid(data=df_plot, x='Rating', y='Revenue (Millions)',
                   height=9, ratio=4, space=0.15)

# 主图：六边形热力图
g.plot_joint(
    sns.histplot,
    bins=(18, 20),    # 注意新版 seaborn 参数名可能是 bin 相关
    cmap='YlOrRd',
    pmax=0.85,
    cbar=True,
    cbar_kws={'label': 'Number of Movies', 'shrink': 0.75},
    edgecolor='white', linewidth=0.3
)

# 改用 scatter + hexbin 方式
g.ax_joint.clear()
# Hexbin plot
hb = g.ax_joint.hexbin(df_plot['Rating'], df_plot['Revenue (Millions)'],
                        gridsize=25, cmap='YlOrRd', mincnt=1,
                        edgecolor='white', linewidth=0.2, alpha=0.9)
cbar_ax = fig.add_axes([0.92, 0.12, 0.02, 0.6]) if False else None
# simpler version
from mpl_toolkits.axes_grid1 import make_axes_locatable
divider = make_axes_locatable(g.ax_joint)
cax = divider.append_axes('right', size='5%', pad=0.1)
plt.colorbar(hb, cax=cax, label='Count')

# 添加回归线
sns.regplot(x='Rating', y='Revenue (Millions)', data=df_plot,
            scatter=False, ax=g.ax_joint,
            line_kws={'color': C['navy'], 'linewidth': 3, 'linestyle': '--'})

# 边缘直方图
sns.histplot(data=df_plot, x='Rating', bins=25, color=C['blue'], alpha=0.6, ax=g.ax_marg_x)
sns.histplot(data=df_plot, y='Revenue (Millions)', bins=25, color=C['coral'], alpha=0.6, ax=g.ax_marg_y)

# 相关系数标注
g.ax_joint.text(0.05, 0.93,
                f"Pearson r = {corr_rr:.3f}\n(moderate positive)",
                transform=g.ax_joint.transAxes, fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#bbb', alpha=0.9))

# 高票房标注
top3 = df_plot.nlargest(3, 'Revenue (Millions)')
for _, m in top3.iterrows():
    g.ax_joint.annotate(m['Title'][:25] + '...',
                        (m['Rating'], m['Revenue (Millions)']),
                        xytext=(15, 20), textcoords='offset points',
                        fontsize=7, color=C['slate'],
                        arrowprops=dict(arrowstyle='->', color='gray', lw=0.6))

g.ax_joint.set_xlabel('IMDB 评分', fontsize=13, fontweight='bold')
g.ax_joint.set_ylabel('全球票房（百万美元）', fontsize=13, fontweight='bold')
g.fig.suptitle('电影评分与票房联合分布', fontsize=16, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('5_评分与票房关系.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f"Moderate positive correlation (r = {corr_rr:.3f}) between rating and revenue.")
print("High-rated movies tend to earn more, but the relationship is far from deterministic.")


In [ ]:

# ============ Chart 5 (Fallback): 高级散点图版本 ============
fig, ax = plt.subplots(figsize=(12, 8))

df_plot = df_clean[df_clean['Revenue (Millions)'] > 0]
corr_rr = df_plot['Rating'].corr(df_plot['Revenue (Millions)'])

# 散点（颜色 = Metascore）
sc = ax.scatter(df_plot['Rating'], df_plot['Revenue (Millions)'],
                c=df_plot['Metascore'], cmap='RdYlGn',
                alpha=0.55, s=55, edgecolor='white', linewidth=0.3)

# 回归线
sns.regplot(x='Rating', y='Revenue (Millions)', data=df_plot,
            scatter=False, ax=ax,
            line_kws={'color': C['navy'], 'linewidth': 3, 'linestyle': '--'})

# 颜色条
cbar = plt.colorbar(sc, ax=ax, shrink=0.82)
cbar.set_label('媒体评分', fontsize=10)

# 相关系数
ax.text(0.04, 0.92,
        f"Pearson r = {corr_rr:.3f}",
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#ccc', alpha=0.9))

# 高票房标注
top5 = df_plot.nlargest(5, 'Revenue (Millions)')
for _, m in top5.iterrows():
    ax.annotate(m['Title'][:30],
                (m['Rating'], m['Revenue (Millions)']),
                xytext=(8, 12), textcoords='offset points',
                fontsize=7, color=C['slate'],
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5, alpha=0.6))

# 横纵坐标轴标签
ax.set_xlabel('IMDB 评分', fontsize=13, fontweight='bold')
ax.set_ylabel('全球票房（百万美元）', fontsize=13, fontweight='bold')
ax.set_title('电影评分与票房关系（颜色 = 媒体评分）', fontsize=16, fontweight='bold', pad=15)

sns.despine()
plt.tight_layout()
plt.savefig('5_评分与票房关系.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f"Rating-Revenue correlation: r = {corr_rr:.3f}")


### 5.6 各年份电影评分对比

**分析问题**：不同年份的电影评分是否存在系统性差异？


In [ ]:

# ============ Chart 6: 各年度评分箱线图（增强版） ============
fig, ax = plt.subplots(figsize=(14, 7.5))

years_sorted = sorted(df_clean['Year'].unique())
yearly_data = [df_clean[df_clean['Year'] == y]['Rating'].values for y in years_sorted]
n_years = len(years_sorted)

# 箱线图
bp = ax.boxplot(yearly_data, patch_artist=True, widths=0.55,
                medianprops={'color': C['navy'], 'linewidth': 2.5},
                whiskerprops={'linewidth': 1.5},
                capprops={'linewidth': 1.5},
                flierprops={'marker': 'o', 'markerfacecolor': C['coral'],
                           'markersize': 4, 'alpha': 0.35})

# 渐变色
colors_box = sns.color_palette("vlag", n_colors=n_years)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

# 年均值折线
year_means = df_clean.groupby('Year')['Rating'].mean()
x_positions = range(1, n_years + 1)
ax.plot(x_positions, year_means.values,
        'D-', color=C['coral'], linewidth=3, markersize=9,
        markerfacecolor='white', markeredgewidth=2.5, markeredgecolor=C['coral'],
        label='年度平均评分', zorder=10)

# 均值数据标签
for xi, yi in zip(x_positions, year_means.values):
    ax.annotate(f'{yi:.2f}', (xi, yi),
                textcoords="offset points", xytext=(0, -18),
                ha='center', fontsize=8, fontweight='bold', color=C['coral'])

# 显示每年样本量
year_counts = df_clean['Year'].value_counts().sort_index()
for xi, cnt in zip(x_positions, year_counts.values):
    ax.text(xi, 1.2, f'n={cnt}', ha='center', fontsize=7.5, color='gray')

# 横纵坐标轴标签
ax.set_xlabel('上映年份', fontsize=13, fontweight='bold')
ax.set_ylabel('IMDB 评分', fontsize=13, fontweight='bold')
ax.set_title('各年度电影评分分布（2006-2016）', fontsize=16, fontweight='bold', pad=15)
ax.set_xticklabels(years_sorted, rotation=40, ha='right', fontsize=10)
ax.set_ylim(0, 10)
ax.legend(fontsize=11, loc='lower left', frameon=True)

sns.despine()
plt.tight_layout()
plt.savefig('6_各年份评分箱线图.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f"Highest avg rating: {year_means.idxmax():.0f} ({year_means.max():.2f})")
print(f"Lowest avg rating:  {year_means.idxmin():.0f} ({year_means.min():.2f})")


### 5.7 高产导演 TOP10

**分析问题**：谁是最高产的导演？产量与质量是否冲突？


In [ ]:

# ============ Chart 7: 导演TOP10（气泡棒棒糖图） ============
fig, ax = plt.subplots(figsize=(13, 7.5))

dir_count = df_clean['Director'].value_counts().head(10)
dir_rating = df_clean.groupby('Director')['Rating'].agg(['mean', 'std', 'count'])
dir_rating = dir_rating.loc[dir_count.index]

colors_dir = sns.color_palette("Blues_r", n_colors=10)

# 棒棒糖
for i, (val, color) in enumerate(zip(dir_count.values, colors_dir)):
    ax.plot([0, val], [i, i], '-', color=color, linewidth=3, alpha=0.65, zorder=2)

# 气泡（大小编码平均评分）
sizes = dir_rating['mean'] * 80
sc = ax.scatter(dir_count.values, range(len(dir_count)),
                s=sizes, c=dir_rating['mean'], cmap='RdYlGn',
                edgecolor='white', linewidth=1.5, zorder=5, vmin=6.0, vmax=8.0)

# 颜色条
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
cbar.set_label('平均评分', fontsize=10)

# 数据标签（数量 + 均分）
for i, (director, cnt, avg_r) in enumerate(zip(dir_count.index, dir_count.values, dir_rating['mean'])):
    ax.text(cnt + 0.35, i, f'{cnt} films  |  avg {avg_r:.1f}',
            va='center', fontsize=10.5, fontweight='bold', color=C['navy'])

ax.set_yticks(range(len(dir_count)))
ax.set_yticklabels(dir_count.index, fontsize=11)
ax.invert_yaxis()

# 横纵坐标轴标签
ax.set_xlabel('执导电影数量（部）', fontsize=13, fontweight='bold')
ax.set_ylabel('导演', fontsize=13, fontweight='bold')
ax.set_title('高产导演 TOP 10（气泡大小 = 平均评分）', fontsize=15, fontweight='bold', pad=15)
ax.set_xlim(0, dir_count.values.max() * 1.45)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig('7_导演TOP10.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f"Top director: {dir_count.index[0]} ({dir_count.values[0]} films, avg rating {dir_rating['mean'].iloc[0]:.1f})")


### 5.8 数值特征相关性热力图

**分析问题**：各数值变量之间存在怎样的关联结构？


In [ ]:

# ============ Chart 8: 相关性热力图（高级版） ============
fig, ax = plt.subplots(figsize=(11, 9))

corr_cols = ['Year', 'Runtime (Minutes)', 'Rating', 'Votes',
             'Revenue (Millions)', 'Metascore']
corr_labels = ['Year', 'Runtime\n(min)', 'Rating', 'Votes',
               'Revenue\n(Millions)', 'Metascore']
corr_matrix = df_clean[corr_cols].corr()

# 半三角遮罩
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# 高级发散配色
cmap = sns.diverging_palette(250, 15, s=80, l=45, n=15, center='light')

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f',
            cmap=cmap, center=0, square=True,
            linewidths=1.8, linecolor='white',
            cbar_kws={'shrink': 0.75, 'label': 'Pearson Correlation Coefficient'},
            annot_kws={'fontsize': 12, 'fontweight': 'bold'},
            vmin=-1, vmax=1, ax=ax,
            xticklabels=corr_labels, yticklabels=corr_labels)

ax.set_xticklabels(corr_labels, fontsize=11, rotation=25, ha='right')
ax.set_yticklabels(corr_labels, fontsize=11, rotation=0)
ax.set_title('数值特征相关性热力图', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('8_相关性热力图.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print("Strongest correlations:")
# 展平找最大的几个
corr_flat = corr_matrix.where(mask).stack().abs().sort_values(ascending=False)
for (a, b), v in corr_flat.head(5).items():
    print(f"  {a} vs {b}: r = {corr_matrix.loc[a, b]:.3f}")


### 5.9 主要电影类型：口碑 vs 票房

**分析问题**：不同类型电影在口碑和商业表现上有何差异？


In [ ]:

# ============ Chart 9: 类型评分 vs 票房（气泡图） ============
fig, ax = plt.subplots(figsize=(13, 8.5))

major_genres = genre_count.head(12).index
type_data = []
for g in major_genres:
    mask = df_clean['Genre'].str.contains(g, na=False)
    subset = df_clean[mask]
    type_data.append({
        'Genre': g,
        'Avg Rating': subset['Rating'].mean(),
        'Avg Revenue': subset['Revenue (Millions)'].mean(),
        'Count': mask.sum(),
        'Median Revenue': subset['Revenue (Millions)'].median(),
    })
tdf = pd.DataFrame(type_data)

# 气泡图
sc = ax.scatter(tdf['Avg Rating'], tdf['Avg Revenue'],
                s=tdf['Count'] * 1.5,
                c=tdf['Avg Rating'], cmap='RdYlGn',
                alpha=0.78, edgecolor='white', linewidth=1.5,
                zorder=10, vmin=6.0, vmax=7.5)

# 颜色条
cbar = plt.colorbar(sc, ax=ax, shrink=0.82)
cbar.set_label('平均评分', fontsize=10)

# 类型标签（智能偏移避免重叠）
offsets = [
    (10, 8), (-15, 12), (10, -12), (-12, -8),
    (12, 6), (-14, 10), (8, -10), (-10, 12),
    (14, -8), (-8, -14), (10, -14), (-14, -10)
]
for i, (_, row) in enumerate(tdf.iterrows()):
    ox, oy = offsets[i % len(offsets)]
    ax.annotate(row['Genre'],
                (row['Avg Rating'], row['Avg Revenue']),
                xytext=(ox, oy), textcoords='offset points',
                fontsize=10.5, fontweight='bold', color=C['navy'],
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8, alpha=0.6))

# 参考线
ax.axhline(tdf['Avg Revenue'].median(), color='gray', linestyle=':',
           linewidth=1, alpha=0.5)
ax.axvline(tdf['Avg Rating'].median(), color='gray', linestyle=':',
           linewidth=1, alpha=0.5)

# 象限标注
mid_x, mid_y = tdf['Avg Rating'].median(), tdf['Avg Revenue'].median()
ax.text(tdf['Avg Rating'].max()-0.05, tdf['Avg Revenue'].max()*0.97,
        'High Rating\\nHigh Revenue', fontsize=8, color=C['teal'], ha='right')
ax.text(tdf['Avg Rating'].min()+0.05, tdf['Avg Revenue'].max()*0.97,
        'Low Rating\\nHigh Revenue', fontsize=8, color=C['gold'], ha='left')
ax.text(tdf['Avg Rating'].max()-0.05, tdf['Avg Revenue'].min()*1.05,
        'High Rating\\nLow Revenue', fontsize=8, color=C['purple'], ha='right')
ax.text(tdf['Avg Rating'].min()+0.05, tdf['Avg Revenue'].min()*1.05,
        'Low Rating\\nLow Revenue', fontsize=8, color=C['coral'], ha='left')

# 横纵坐标轴标签
ax.set_xlabel('平均 IMDB 评分', fontsize=13, fontweight='bold')
ax.set_ylabel('平均票房（百万美元）', fontsize=13, fontweight='bold')
ax.set_title('主要电影类型：评分 vs 票房（气泡大小 = 电影数量）',
             fontsize=15, fontweight='bold', pad=15)

sns.despine()
plt.tight_layout()
plt.savefig('9_类型评分票房气泡图.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print("Sci-Fi & Adventure: highest avg revenue (blockbuster types).")
print("Biography & History: higher avg rating but modest revenue (critically acclaimed).")
print("Horror: lower on both dimensions (niche audience).")


## 6. 主要结论

---

### 市场格局
- **Drama、Action、Comedy** 三大类型合计占比超 50%，构成电影市场的主体
- 小众类型（Western、Musical 等）产量极低，属于细分市场

### 产业趋势
- 2006—2016 年电影产量呈**加速增长**态势，从 44 部跃升至 297 部（数据集中）
- 反映全球电影产业在过去十年的快速扩张

### 口碑与票房
- 评分与票房呈**中等正相关**（Pearson r ≈ 0.24），口碑好确实有助于票房
- 但相关性远非完美：**高分段票房离散度极大**，「叫好不叫座」与「叫座不叫好」现象并存
- **Votes（关注度）才是票房的最强预测因子** —— 关注度比口碑更能解释票房差异

### 时长规律
- 电影时长近似**正态分布**，中心约 113 分钟，90% 落在 88—150 分钟
- 这一区间切合影院排片需求和观众观影习惯

### 年份差异
- 年度间平均评分波动很小（6.44—7.13），**电影工业的创作标准在十年间保持稳定**
- 2016 年评分略低可能与该年样本量大、类型更多元有关

### 创作者
- TOP10 高产导演平均评分集中在 6.5—7.5 区间，**高产量与高质量并不矛盾**
- Ridley Scott 以 8 部作品居首

### 相关性结构
- **用户评分 ↔ 媒体评分**（r ≈ 0.63）：专业与大众品味总体一致
- **票房 ↔ 投票数**（r ≈ 0.63）：高关注度驱动高票房
- **时长**与其他变量均弱相关：电影长度独立于评分、票房和年份

---

> **局限性**：本数据集为 IMDB 样本非全量；票房为全球票房不含衍生收入；少数类型样本量小，结论外推需谨慎。


In [ ]:

print("=" * 60)
print("Report Summary")
print("=" * 60)
print(f"Dataset: {df_clean.shape[0]} movies x {df_clean.shape[1]} columns")
print(f"Charts generated: 9")
print(f"  1. Top 15 Genres (Lollipop Chart)")
print(f"  2. Annual Movie Trend (Area + Trend Line)")
print(f"  3. Runtime Distribution (Histogram + KDE + Rug)")
print(f"  4. Rating Distribution (Histogram + Violin/Box)")
print(f"  5. Rating vs Revenue (Hexbin Scatter + Regression)")
print(f"  6. Rating by Year (Enhanced Box Plot)")
print(f"  7. Top 10 Directors (Bubble Lollipop)")
print(f"  8. Correlation Matrix (Heatmap)")
print(f"  9. Genre: Rating vs Revenue (Bubble Chart)")
print()
print("All charts saved as high-resolution PNG files (200 DPI).")
print("All axis labels configured for clear Chinese/English display.")
